# Typo detection from causal-language-model surprisal

A causal LM estimates (p(x_t\mid x_{<t})). Token surprisal, (-\log p(x_t\mid x_{<t})), is used as an anomaly signal. The experiment separates calibration from final evaluation because choosing a threshold on the test set would inflate accuracy.

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve, auc

from nlp_portfolio.generation import token_surprisals, calibrate_threshold

MODEL_ID = "gpt2"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()
print("device:", device)

In [ ]:
examples = [
    "The researcher published the results yesterday.",
    "The researhcer published the results yesterday.",
]
for sentence in examples:
    scores = token_surprisals(sentence, tokenizer, model)
    print(sentence)
    print([(token, round(value, 2)) for token, value in scores])

The logit at position (t-1) predicts token (t). Comparing a token to the logit at its own position leaks that token into the representation and yields an invalid score.

## Labeled data and sentence scores

Maximum surprisal is sensitive to one strong anomaly. The mean of the largest two token surprisals is more stable for multi-token misspellings.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/elenipapadopulos/NLP_LAB3_Datasets/main/typo_dataset1.csv"
data = pd.read_csv(DATA_URL)

def sentence_score(text, top_n=2):
    values = sorted((value for _, value in token_surprisals(text, tokenizer, model)), reverse=True)
    return float(np.mean(values[:top_n])) if values else 0.0

data["score"] = [sentence_score(text) for text in data["text"]]
data[["text", "label", "score"]].head()

In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(data)), test_size=0.35, random_state=42, stratify=data["label"]
)
threshold = calibrate_threshold(data.loc[train_idx, "score"], data.loc[train_idx, "label"])
test_prediction = (data.loc[test_idx, "score"] >= threshold).astype(int)
print("calibration threshold:", round(threshold, 3))
print(classification_report(data.loc[test_idx, "label"], test_prediction, digits=3))

In [ ]:
precision, recall, _ = precision_recall_curve(data.loc[test_idx, "label"], data.loc[test_idx, "score"])
print("held-out PR-AUC:", round(auc(recall, precision), 3))

audit = data.loc[test_idx, ["text", "label", "score"]].copy()
audit["prediction"] = test_prediction
audit["correct"] = audit["label"] == audit["prediction"]
audit.sort_values(["correct", "score"]).head(12)

Rare names, scientific terminology and unusual syntax can be surprising without containing typos. Common real-word errors can remain probable in context. This detector is therefore an uncertainty signal, not a spelling oracle. A practical system should combine it with edit-distance candidates and a domain lexicon, then report recall at an operational false-positive rate.